In [ ]:
import numpy as np, pandas as pd, requests
import pyarrow as pa, pyarrow.parquet as pq
import glob, io, os, re, shutil, time, zipfile
from pathlib import Path
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
 

BASE_DIR   = Path("")

RAW_DIR = BASE_DIR / "Data" / "Raw"
INTERIM_DIR = BASE_DIR / "Data" / "Interim"
PROCESSED_DIR = BASE_DIR / "Data" / "Processed"

# GPREG

In [2]:
RAW_DIR_GPREG = RAW_DIR / "GP_REG"
RAW_DIR_GPREG.mkdir(parents=True, exist_ok=True)

INTERIM_DIR_GPREG = INTERIM_DIR / "GP_REG"
INTERIM_DIR_GPREG.mkdir(parents=True, exist_ok=True)

## Download

In [3]:
HUB_URL = "https://digital.nhs.uk/data-and-information/publications/statistical/patients-registered-at-a-gp-practice"

MONTH_NUMBERS = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
}


def slug_month(slug):
    """Return the month number found in a slug, or None if there isn't one."""
    s = slug.lower()
    for name, num in MONTH_NUMBERS.items():
        if name in s:
            return num
    return None


def download_patients(year, start_month=1, end_month=12):
    year = str(year)
    output_dir = RAW_DIR_GPREG / year
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Fetching publication indexes for {year}...")
    hub_response = requests.get(HUB_URL, timeout=60)
    hub_response.raise_for_status()
    soup = BeautifulSoup(hub_response.content, "html.parser")

    # 1. Collect the monthly landing pages for this year
    monthly_pages = []
    for link in soup.find_all("a", href=True):
        link_url = link["href"]
        if link_url.startswith("#"):
            continue
        if "patients-registered-at-a-gp-practice" in link_url and f"-{year}" in link_url:
            full_page_url = urljoin(HUB_URL, link_url)
            url_slug = link_url.split("/")[-1]
            if full_page_url not in [p[1] for p in monthly_pages]:
                monthly_pages.append((url_slug, full_page_url))

    print(f"Found {len(monthly_pages)} months for {year}.")

    # 2. Keep only the months you asked for
    kept_pages = []
    for slug, page_url in monthly_pages:
        m = slug_month(slug)
        if m is None:
            print(f"Couldn't read a month from '{slug}', keeping it")
            kept_pages.append((slug, page_url))
        elif start_month <= m <= end_month:
            kept_pages.append((slug, page_url))
    monthly_pages = kept_pages
    print(f"{len(monthly_pages)} months left after applying months {start_month}-{end_month}\n")

    failures = []

    # 3. For each month, find the male and female "single year of age (GP practice)" files
    for month_slug, page_url in monthly_pages:
        page_response = requests.get(page_url, timeout=60)
        if page_response.status_code != 200:
            print(f"Couldn't open page for {month_slug} (status {page_response.status_code})")
            failures.append(month_slug)
            continue

        page_soup = BeautifulSoup(page_response.content, "html.parser")
        targets = {"females": None, "males": None}

        for a_tag in page_soup.find_all("a", href=True):
            href = a_tag["href"]
            if href.startswith("#"):
                continue
            parent_text = a_tag.find_parent().get_text() if a_tag.find_parent() else ""
            combined_text = (a_tag.get_text() + " " + parent_text).lower()

            is_syoa = "single year of age" in combined_text and "gp practice" in combined_text
            is_downloadable = href.lower().endswith(".zip") or href.lower().endswith(".csv")

            if is_syoa and is_downloadable:
                if "female" in combined_text and targets["females"] is None:
                    targets["females"] = urljoin(page_url, href)
                elif "male" in combined_text and "female" not in combined_text and targets["males"] is None:
                    targets["males"] = urljoin(page_url, href)

        for sex, file_download_url in targets.items():
            unique_name = f"gp-reg-pat-prac-syoa-{sex}-{month_slug}.csv"
            target_path = output_dir / unique_name

            if target_path.exists():
                print(f"{unique_name} already done, skipping")
                continue

            if not file_download_url:
                print(f"Couldn't find '{sex}' file link for {month_slug}")
                failures.append(f"{month_slug} ({sex})")
                continue

            file_response = requests.get(file_download_url, timeout=120)
            if file_response.status_code != 200:
                print(f"Failed to download {sex} file for {month_slug} (status {file_response.status_code})")
                failures.append(f"{month_slug} ({sex})")
                continue

            if file_download_url.lower().endswith(".zip"):
                try:
                    with zipfile.ZipFile(io.BytesIO(file_response.content)) as z:
                        csvs = [i for i in z.infolist() if i.filename.lower().endswith(".csv")]
                        if not csvs:
                            print(f"No csv inside zip for {month_slug} ({sex})")
                            failures.append(f"{month_slug} ({sex})")
                            continue
                        if len(csvs) > 1:
                            print(f"{len(csvs)} csvs in zip for {month_slug} ({sex}); saving the first")
                        target_path.write_bytes(z.read(csvs[0].filename))
                    print(f"Downloaded & saved (from zip): {unique_name}")
                except zipfile.BadZipFile:
                    print(f"Corrupted zip for {month_slug} ({sex})")
                    failures.append(f"{month_slug} ({sex})")
            else:
                target_path.write_bytes(file_response.content)
                print(f"Downloaded & saved (direct csv): {unique_name}")

    print(f"\nDone. Saved to {output_dir}")
    if failures:
        print("Failed:", failures)

In [4]:
download_patients(2022, start_month=7)  
download_patients(2023)   
download_patients(2024)      
download_patients(2025) 
download_patients(2026, end_month=3)  

Fetching publication indexes for 2022...
Found 12 months for 2022.
6 months left after applying months 7-12

Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-females-december-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-males-december-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-females-november-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-males-november-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-females-october-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-males-october-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-females-september-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-males-september-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-females-august-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-males-august-2022.csv
Downloaded & saved (direct csv): gp-reg-pat-prac-syoa-females-july-2022.csv
Downloaded & saved (direct cs

## Merge

In [5]:
KEEP_COLS = ["ORG_CODE", "AGE", "NUMBER_OF_PATIENTS", "SEX", "EXTRACT_DATE"]


def merge_patients(years=None, out_name="GP_REG_combined.parquet"):
    # Year folders: all of them by default, or just the ones you list
    if years is None:
        input_dirs = sorted(d for d in RAW_DIR_GPREG.iterdir() if d.is_dir() and d.name.isdigit())
    else:
        input_dirs = [RAW_DIR_GPREG / str(y) for y in years]
    print("Year folders:", [d.name for d in input_dirs])

    all_frames = []
    removed_all = 0

    for input_dir in input_dirs:
        csv_files = sorted(glob.glob(os.path.join(input_dir, "gp-reg-pat-prac-syoa-*.csv")))
        print(f"{input_dir.name}: found {len(csv_files)} files")

        for filepath in csv_files:
            filename = os.path.basename(filepath)
            try:
                df = pd.read_csv(
                    filepath,
                    usecols=lambda c: c in KEEP_COLS,
                    dtype={"ORG_CODE": str, "AGE": str, "SEX": str, "EXTRACT_DATE": str},
                    low_memory=False,
                )
            except Exception as e:
                print(f"Failed to read {filename}: {e}")
                continue

            missing = set(KEEP_COLS) - set(df.columns)
            if missing:
                print(f"{filename} is missing columns: {missing}")

            # Drop the practice-level "ALL ages" summary rows per file to save memory
            n = len(df)
            df = df[df["AGE"] != "ALL"]
            removed_all += n - len(df)

            all_frames.append(df)

    if not all_frames:
        print("No files loaded, nothing to save.")
        return

    combined = pd.concat(all_frames, ignore_index=True, sort=False)[KEEP_COLS]
    del all_frames

    print(f"\nRemoved {removed_all:,} 'ALL' summary rows, {len(combined):,} rows remain")
    print(f"Combined shape: {combined.shape}")
    print(f"Dates covered: {combined['EXTRACT_DATE'].nunique()} unique EXTRACT_DATE values")

    dupes = combined.duplicated(["EXTRACT_DATE", "ORG_CODE", "AGE", "SEX"]).sum()
    print(f"Duplicate keys: {dupes:,}")


    output_path = INTERIM_DIR_GPREG / out_name
    combined.to_parquet(output_path, index=False)
    print(f"Saved combined file to {output_path}")

merge_patients()

Year folders: ['2022', '2023', '2024', '2025', '2026']
2022: found 12 files
2023: found 24 files
2024: found 24 files
2025: found 24 files
2026: found 6 files

Removed 567,895 'ALL' summary rows, 54,096,737 rows remain
Combined shape: (54096737, 5)
Dates covered: 45 unique EXTRACT_DATE values
Duplicate keys: 0
Saved combined file to /Users/trinityrose/Desktop/TEST/Data/Interim/GP_REG/GP_REG_combined.parquet


## Processing

### fix dates

In [6]:
merged = pd.read_parquet(INTERIM_DIR_GPREG / "GP_REG_combined.parquet")

# Inspect formats on the unique values, not every row
uniq = pd.Series(merged["EXTRACT_DATE"].astype(str).unique())
print(f"{len(uniq)} unique raw date strings")
print(uniq.str.len().value_counts())
for n in sorted(uniq.str.len().unique()):
    print(f"--- length {n} examples ---")
    print(uniq[uniq.str.len() == n].head(5).tolist())

# Parse the unique values once, then map back
parsed = pd.to_datetime(uniq, format="mixed", dayfirst=True)
mapping = dict(zip(uniq, parsed))
merged["EXTRACT_DATE"] = merged["EXTRACT_DATE"].astype(str).map(mapping)

# Check every raw value mapped to the month you expect
check = pd.DataFrame({"raw": uniq, "parsed": parsed}).sort_values("parsed")
print(check.to_string(index=False))
print("Day-of-month values:", sorted(parsed.dt.day.unique()))
print("Unparsed rows:", merged["EXTRACT_DATE"].isna().sum())

# Restrict to your study window
STUDY_START = pd.Timestamp("2022-07-01")
STUDY_END = pd.Timestamp("2026-03-01")

merged_window = merged[
    (merged["EXTRACT_DATE"] >= STUDY_START) & (merged["EXTRACT_DATE"] <= STUDY_END)
].copy()

print(f"Rows before restriction: {len(merged):,}")
print(f"Rows after restriction: {len(merged_window):,}")
print(f"Date range: {merged_window['EXTRACT_DATE'].min()} to {merged_window['EXTRACT_DATE'].max()}")
print(f"Distinct dates: {merged_window['EXTRACT_DATE'].nunique()} (expect 45 for Jul 2022 to Mar 2026)")

45 unique raw date strings
10    23
9     22
Name: count, dtype: int64
--- length 9 examples ---
['01Aug2022', '01Dec2022', '01-Jul-22', '01Nov2022', '01Oct2022']
--- length 10 examples ---
['2024-08-01', '2024-12-01', '2024-07-01', '2024-06-01', '2024-05-01']
       raw     parsed
 01-Jul-22 2022-07-01
 01Aug2022 2022-08-01
 01Sep2022 2022-09-01
 01Oct2022 2022-10-01
 01Nov2022 2022-11-01
 01Dec2022 2022-12-01
 01Jan2023 2023-01-01
 01Feb2023 2023-02-01
 01Mar2023 2023-03-01
 01Apr2023 2023-04-01
 01May2023 2023-05-01
 01Jun2023 2023-06-01
 01Jul2023 2023-07-01
 01Aug2023 2023-08-01
 01Sep2023 2023-09-01
 01Oct2023 2023-10-01
 01Nov2023 2023-11-01
 01Dec2023 2023-12-01
 01Jan2024 2024-01-01
 01Feb2024 2024-02-01
 01Mar2024 2024-03-01
 01Apr2024 2024-04-01
2024-05-01 2024-05-01
2024-06-01 2024-06-01
2024-07-01 2024-07-01
2024-08-01 2024-08-01
2024-09-01 2024-09-01
2024-10-01 2024-10-01
2024-11-01 2024-11-01
2024-12-01 2024-12-01
2025-01-01 2025-01-01
2025-02-01 2025-02-01
2025-03-01 20

### age banding

In [7]:
# 0. Check what AGE actually contains before converting
ages = merged_window["AGE"].astype(str).unique()
print(sorted(ages, key=lambda a: (not a.isdigit(), int(a) if a.isdigit() else 0, a)))

# 1. Convert the ~100 distinct ages once, not every row
def parse_age(a):
    a = str(a).strip()
    if a.endswith("+"):
        return int(a[:-1])
    return int(a)             # raises on anything unexpected, so you notice

age_map = {a: parse_age(a) for a in ages}
merged_window["AGE_NUM"] = merged_window["AGE"].astype(str).map(age_map).astype("int16")

# 2. Bands via pd.cut (vectorised)
band_order = ['0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39',
              '40-44','45-49','50-54','55-59','60-64','65-69','70-74',
              '75-79','80-84','85-89','90+']
bins = list(range(-1, 90, 5)) + [200]     # -1,4,9,...,89,200
merged_window["AGE_BAND"] = pd.cut(
    merged_window["AGE_NUM"], bins=bins, labels=band_order, right=True
)

print("Unbanded rows:", merged_window["AGE_BAND"].isna().sum())

# 3. Aggregate
banded = (
    merged_window
    .groupby(['ORG_CODE', 'EXTRACT_DATE', 'SEX', 'AGE_BAND'], as_index=False, observed=True)['NUMBER_OF_PATIENTS']
    .sum()
)

banded = banded.sort_values(['ORG_CODE', 'EXTRACT_DATE', 'SEX', 'AGE_BAND']).reset_index(drop=True)

print(f"Total rows after banding: {len(banded):,}")
print(banded['AGE_BAND'].dtype)   # should say category
print(f"Patients before: {merged_window['NUMBER_OF_PATIENTS'].sum():,}  after: {banded['NUMBER_OF_PATIENTS'].sum():,}")
banded.head(40)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95+']
Unbanded rows: 0
Total rows after banding: 10,750,805
category
Patients before: 2,840,664,428  after: 2,840,664,428


,ORG_CODE,EXTRACT_DATE,SEX,AGE_BAND,NUMBER_OF_PATIENTS
0,A81001,2022-07-01,FEMALE,0-4,94
1,A81001,2022-07-01,FEMALE,5-9,125
2,A81001,2022-07-01,FEMALE,10-14,121
3,A81001,2022-07-01,FEMALE,15-19,121
4,A81001,2022-07-01,FEMALE,20-24,116
5,A81001,2022-07-01,FEMALE,25-29,126
6,A81001,2022-07-01,FEMALE,30-34,117
7,A81001,2022-07-01,FEMALE,35-39,120
8,A81001,2022-07-01,FEMALE,40-44,115
9,A81001,2022-07-01,FEMALE,45-49,104


In [8]:
output_path = (PROCESSED_DIR / "GP_REG_processed.parquet")
banded.to_parquet(output_path, index=False)
print(f"Saved banded file to {output_path}")

Saved banded file to /Users/trinityrose/Desktop/TEST/Data/Processed/GP_REG_processed.parquet


# GPWF

In [9]:
RAW_DIR_GPWF = RAW_DIR / "gpwf"
RAW_DIR_GPWF.mkdir(parents=True, exist_ok=True)

INTERIM_DIR_GPWF = INTERIM_DIR / "gpwf"
INTERIM_DIR_GPWF.mkdir(parents=True, exist_ok=True)

## Download

In [10]:
HUB_URL = "https://digital.nhs.uk/data-and-information/publications/statistical/general-and-personal-medical-services"

MONTH_NUMBERS = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
}

MONTH_PAGE_RE = re.compile(r"/general-and-personal-medical-services/(\d{1,2})-([a-z]+)-(\d{4})/?$")
FILE_DATE_RE = re.compile(
    r"(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})",
    re.I,
)

REMAP_CUTOFF = (2022, 7)

def file_period(name):
    """(year, month) found in a filename, or None."""
    m = FILE_DATE_RE.search(name)
    return (int(m.group(2)), MONTH_NUMBERS[m.group(1).lower()]) if m else None


def get_month_pages():
    """{(year, month): page_url} for every monthly release on the hub page."""
    r = requests.get(HUB_URL, timeout=60)
    r.raise_for_status()
    soup = BeautifulSoup(r.content, "html.parser")
    pages = {}
    for link in soup.find_all("a", href=True):
        m = MONTH_PAGE_RE.search(urlparse(link["href"]).path)
        if m and m.group(2) in MONTH_NUMBERS:
            pages[(int(m.group(3)), MONTH_NUMBERS[m.group(2)])] = urljoin(HUB_URL, link["href"])
    return pages


def find_practice_zips(page_url):
    """Every practice-level zip on a release page (some pages have more than one)."""
    r = requests.get(page_url, timeout=60)
    r.raise_for_status()
    soup = BeautifulSoup(r.content, "html.parser")
    urls = []
    for a in soup.find_all("a", href=True):
        text = a.get_text(" ").lower()
        if ("practice-level" in text or "practice level" in text) and a["href"].lower().endswith(".zip"):
            u = urljoin(page_url, a["href"])
            if u not in urls:
                urls.append(u)
    return urls


def download_gpwf(year, start_month=1, end_month=12):
    year = int(year)
    output_dir = RAW_DIR_GPWF / str(year)
    output_dir.mkdir(parents=True, exist_ok=True)

    pages = get_month_pages()
    failures = []

    # Work out which release page each wanted month comes from
    wanted_by_page = {}  # page_url -> {(year, month), ...}
    for month in range(start_month, end_month + 1):
        period = (year, month)
        source = REMAP_CUTOFF if period <= REMAP_CUTOFF else period
        if source not in pages:
            print(f"No release page found for {year}-{month:02d} (not published yet?)")
            failures.append(f"{year}-{month:02d}")
            continue
        wanted_by_page.setdefault(pages[source], set()).add(period)

    already = {file_period(p.name) for p in output_dir.glob("*.csv")}

    for page_url, wanted in wanted_by_page.items():
        slug = page_url.rsplit("/", 1)[-1]
        todo = wanted - already
        if not todo:
            print(f"{slug}: months already downloaded")
            continue

        for zip_url in find_practice_zips(page_url):
            resp = requests.get(zip_url, timeout=300)
            if resp.status_code != 200:
                print(f" {slug}: zip failed (status {resp.status_code}) {zip_url}")
                failures.append(f"{slug} zip")
                continue
            try:
                with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                    for info in z.infolist():
                        name = Path(info.filename).name
                        if not name.lower().endswith(".csv") or "definition" in name.lower():
                            continue
                        if file_period(name) not in todo:  
                            continue
                        (output_dir / name).write_bytes(z.read(info.filename))
                        print(f"saved {name}")
            except zipfile.BadZipFile:
                print(f"{slug}: corrupted zip {zip_url}")
                failures.append(f"{slug} zip")

    have = {file_period(p.name) for p in output_dir.glob("*.csv")}
    missing = [f"{year}-{m:02d}" for m in range(start_month, end_month + 1) if (year, m) not in have]
    print(f"\nDone. Saved to {output_dir}")
    print("Still missing:", missing or "none")
    if failures:
        print("Failures:", failures)

In [11]:
download_gpwf(2022, start_month=7)
download_gpwf(2023)
download_gpwf(2024)
download_gpwf(2025)
download_gpwf(2026, end_month=3)

saved 35. General Practice – July 2022 Practice Level.csv
saved 36. General Practice – August 2022 Practice Level.csv
saved 37. General Practice – September 2022 Practice Level.csv
saved 38. General Practice – October 2022 Practice Level.csv
saved 39. General Practice – November 2022 Practice Level.csv
saved 40. General Practice – December 2022 Practice Level.csv

Done. Saved to /Users/trinityrose/Desktop/TEST/Data/Raw/gpwf/2022
Still missing: none
saved 3 General Practice – January 2023 Practice Level - High level.csv
saved 1 General Practice – January 2023 Practice Level - Detailed.csv
saved 1 General Practice – February 2023 Practice Level - Detailed.csv
saved 3 General Practice – February 2023 Practice Level - High level.csv
saved 3 General Practice – March 2023 Practice Level - High level.csv
saved 1 General Practice – March 2023 Practice Level - Detailed.csv
saved 3 General Practice – April 2023 Practice Level - High level.csv
saved 1 General Practice – April 2023 Practice Level 

## Merge

In [12]:
MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

rows = []
for f in sorted(RAW_DIR_GPWF.glob("*/*.csv")):
    name = f.name
    m = re.search(r"(" + "|".join(MONTHS) + r")\s+(\d{4})", name)
    month, year = (m.group(1), int(m.group(2))) if m else (None, None)

    if "high level" in name.lower():
        kind = "High level"
    elif "detailed" in name.lower():
        kind = "Detailed"
    elif "practice level" in name.lower():
        kind = "Practice level (pre-2023)"
    else:
        kind = "Other"

    rows.append({"folder": f.parent.name, "file": name, "month": month,
                 "year": year, "kind": kind,
                 "size_kb": round(f.stat().st_size / 1024)})

inventory = pd.DataFrame(rows)

# 1. Every file, grouped by folder
for folder, g in inventory.groupby("folder"):
    print(f"\n=== {folder}: {len(g)} files ===")
    for _, r in g.iterrows():
        print(f"  {r['month'] or '??':<10} {r['kind']:<26} {r['size_kb']:>6} KB  {r['file']}")

# 2. Months present per year, by file type
summary = (inventory.dropna(subset=["month"])
           .pivot_table(index="month", columns=["year", "kind"],
                        values="file", aggfunc="count", fill_value=0)
           .reindex(MONTHS))
print("\n=== Files per month (rows) by year and type (columns) ===")
print(summary.to_string())

# 3. Missing months per folder
print("\n=== Missing months ===")
for folder, g in inventory.groupby("folder"):
    have = set(g["month"].dropna())
    missing = [m for m in MONTHS if m not in have]
    print(f"  {folder}: {', '.join(missing) if missing else 'none'}")

# 4. Files whose names I couldn't parse (these will break your merge cell)
bad = inventory[inventory["month"].isna() | (inventory["kind"] == "Other")]
if not bad.empty:
    print("\n=== Unrecognised files ===")
    print(bad[["folder", "file"]].to_string(index=False))


=== 2022: 6 files ===
  July       Practice level (pre-2023)   12427 KB  35. General Practice – July 2022 Practice Level.csv
  August     Practice level (pre-2023)   12425 KB  36. General Practice – August 2022 Practice Level.csv
  September  Practice level (pre-2023)   12425 KB  37. General Practice – September 2022 Practice Level.csv
  October    Practice level (pre-2023)   12398 KB  38. General Practice – October 2022 Practice Level.csv
  November   Practice level (pre-2023)   12375 KB  39. General Practice – November 2022 Practice Level.csv
  December   Practice level (pre-2023)   12439 KB  40. General Practice – December 2022 Practice Level.csv

=== 2023: 24 files ===
  April      Detailed                    13746 KB  1 General Practice – April 2023 Practice Level - Detailed.csv
  August     Detailed                    13745 KB  1 General Practice – August 2023 Practice Level - Detailed.csv
  December   Detailed                    13745 KB  1 General Practice – December 2023 Prac

In [13]:
keep = sorted(
    f for f in RAW_DIR_GPWF.glob("*/*.csv")
    if "high level" not in f.name.lower()
    and file_period(f.name) is not None
    and (2022, 1) <= file_period(f.name) <= (2026, 8)
)

periods = [file_period(f.name) for f in keep]
assert len(periods) == len(set(periods)), "more than one file for a month"
print(len(keep))   # should be 56 (Jan 2022 to Aug 2026)

45


In [14]:
import pandas as pd
from collections import defaultdict

keep = sorted(keep, key=lambda f: file_period(f.name))

cols_by_period = {file_period(f.name): list(pd.read_csv(f, nrows=0).columns) for f in keep}
periods = sorted(cols_by_period)
all_cols = sorted(set().union(*cols_by_period.values()))
common = set.intersection(*(set(c) for c in cols_by_period.values()))

print(f"{len(periods)} files, {len(all_cols)} distinct columns, {len(common)} in every file\n")

# Columns that aren't in every file, grouped by when they appear
groups = defaultdict(list)
for c in all_cols:
    if c in common:
        continue
    present = [p for p in periods if c in cols_by_period[p]]
    groups[(present[0], present[-1], len(present))].append(c)

for (first, last, n), cols in sorted(groups.items()):
    print(f"{first[0]}-{first[1]:02d} to {last[0]}-{last[1]:02d} ({n} files): {len(cols)} cols, e.g. {cols[:3]}")

print("\nFirst columns, earliest file:", cols_by_period[periods[0]][:20])
print("First columns, latest file:  ", cols_by_period[periods[-1]][:20])

45 files, 829 distinct columns, 697 in every file

2022-07 to 2024-03 (21 files): 24 cols, e.g. ['FEMALE_DPC_HI_THERA_FTE', 'FEMALE_DPC_HI_THERA_HC', 'FEMALE_DPC_PSYCH_WELLB_PRAC_FTE']
2022-12 to 2026-03 (40 files): 6 cols, e.g. ['FEMALE_DPC_DIETICIAN_FTE', 'FEMALE_DPC_DIETICIAN_HC', 'MALE_DPC_DIETICIAN_FTE']
2023-01 to 2026-03 (39 files): 47 cols, e.g. ['TOTAL_ADMIN_FTE_25TO29', 'TOTAL_ADMIN_FTE_30TO34', 'TOTAL_ADMIN_FTE_35TO39']
2023-04 to 2026-03 (36 files): 6 cols, e.g. ['FEMALE_DPC_GPA_FTE', 'FEMALE_DPC_GPA_HC', 'MALE_DPC_GPA_FTE']
2023-05 to 2026-03 (35 files): 6 cols, e.g. ['FEMALE_DPC_APP_PHYSICIAN_ASSOC_FTE', 'FEMALE_DPC_APP_PHYSICIAN_ASSOC_HC', 'MALE_DPC_APP_PHYSICIAN_ASSOC_FTE']
2023-12 to 2026-03 (28 files): 6 cols, e.g. ['FEMALE_ADMIN_DT_LEAD_FTE', 'FEMALE_ADMIN_DT_LEAD_HC', 'MALE_ADMIN_DT_LEAD_FTE']
2024-03 to 2026-03 (25 files): 1 cols, e.g. ['TOTAL_UNKNOWN_GENDER']
2024-04 to 2026-03 (24 files): 30 cols, e.g. ['FEMALE_DPC_ADV_DIETICIAN_PRAC_FTE', 'FEMALE_DPC_ADV_DIETICI

In [15]:
id_cols = ["PRAC_CODE", "PRAC_NAME", "PCN_CODE", "PCN_NAME", "SUB_ICB_CODE",
           "SUB_ICB_NAME", "ICB_CODE", "ICB_NAME", "REGION_CODE", "REGION_NAME"]
print("ID columns not in every file:", [c for c in id_cols if c not in common] or "none")

gp_fte_cols = sorted({c for cols in cols_by_period.values() for c in cols
                      if c.startswith("TOTAL_GP") and c.endswith("_FTE")})

for c in gp_fte_cols:
    n = sum(c in cols for cols in cols_by_period.values())
    print(f"{c:<32} in {n} of {len(periods)} files")

ID columns not in every file: none
TOTAL_GP_EXL_FTE                 in 45 of 45 files
TOTAL_GP_EXTGL_FTE               in 45 of 45 files
TOTAL_GP_EXTG_FTE                in 45 of 45 files
TOTAL_GP_FTE                     in 45 of 45 files
TOTAL_GP_LOCUM_ABS_FTE           in 45 of 45 files
TOTAL_GP_LOCUM_OTH_FTE           in 45 of 45 files
TOTAL_GP_LOCUM_VAC_FTE           in 45 of 45 files
TOTAL_GP_PTNR_PROV_FTE           in 45 of 45 files
TOTAL_GP_RET_FTE                 in 45 of 45 files
TOTAL_GP_SAL_BY_OTH_FTE          in 45 of 45 files
TOTAL_GP_SAL_BY_PRAC_FTE         in 45 of 45 files
TOTAL_GP_SEN_PTNR_FTE            in 45 of 45 files
TOTAL_GP_TRN_GR_F1_2_FTE         in 45 of 45 files
TOTAL_GP_TRN_GR_OTH_FTE          in 45 of 45 files
TOTAL_GP_TRN_GR_ST1_FTE          in 45 of 45 files
TOTAL_GP_TRN_GR_ST2_FTE          in 45 of 45 files
TOTAL_GP_TRN_GR_ST3_FTE          in 45 of 45 files
TOTAL_GP_TRN_GR_ST4_FTE          in 45 of 45 files


In [16]:
use_cols = id_cols + gp_fte_cols

dfs = []
for f in keep:
    y, m = file_period(f.name)
    df = pd.read_csv(f, usecols=use_cols, dtype={c: str for c in id_cols})
    df.insert(0, "YEAR_MONTH", f"{y}{m:02d}")
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

print(final_df.shape)
print("months:", final_df["YEAR_MONTH"].nunique())
print("duplicate practice-months:", final_df.duplicated(["YEAR_MONTH", "PRAC_CODE"]).sum())
print(final_df.isna().sum()[lambda s: s > 0])


(283247, 29)
months: 45
duplicate practice-months: 0
SUB_ICB_NAME                  15
ICB_CODE                      15
ICB_NAME                      15
REGION_CODE                   15
REGION_NAME                   15
TOTAL_GP_FTE                2832
TOTAL_GP_EXTG_FTE           2832
TOTAL_GP_EXL_FTE            2832
TOTAL_GP_EXTGL_FTE          2832
TOTAL_GP_SEN_PTNR_FTE       2832
TOTAL_GP_PTNR_PROV_FTE      2832
TOTAL_GP_SAL_BY_PRAC_FTE    2832
TOTAL_GP_SAL_BY_OTH_FTE     2832
TOTAL_GP_RET_FTE            2832
TOTAL_GP_LOCUM_VAC_FTE      2832
TOTAL_GP_LOCUM_ABS_FTE      2832
TOTAL_GP_LOCUM_OTH_FTE      2832
dtype: int64


In [17]:
for c in gp_fte_cols:
    as_num = pd.to_numeric(final_df[c], errors="coerce")
    bad = final_df.loc[as_num.isna() & final_df[c].notna(), ["YEAR_MONTH", c]]
    if len(bad):
        print(f"{c}: {len(bad)} non-numeric values")
        print(bad[c].value_counts().head(), "\n")
        print(bad["YEAR_MONTH"].value_counts().sort_index().head(), "\n")
    final_df[c] = as_num

out_path = INTERIM_DIR_GPWF / "gpwf_combined.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(out_path, index=False)

TOTAL_GP_EXL_FTE: 217 non-numeric values
TOTAL_GP_EXL_FTE
ND    217
Name: count, dtype: int64 

YEAR_MONTH
202207    25
202208    26
202209    31
202210    33
202211    35
Name: count, dtype: int64 

TOTAL_GP_EXTGL_FTE: 219 non-numeric values
TOTAL_GP_EXTGL_FTE
ND    219
Name: count, dtype: int64 

YEAR_MONTH
202207    25
202208    26
202209    31
202210    33
202211    35
Name: count, dtype: int64 

TOTAL_GP_EXTG_FTE: 219 non-numeric values
TOTAL_GP_EXTG_FTE
ND    219
Name: count, dtype: int64 

YEAR_MONTH
202207    25
202208    26
202209    31
202210    33
202211    35
Name: count, dtype: int64 

TOTAL_GP_FTE: 217 non-numeric values
TOTAL_GP_FTE
ND    217
Name: count, dtype: int64 

YEAR_MONTH
202207    25
202208    26
202209    31
202210    33
202211    35
Name: count, dtype: int64 

TOTAL_GP_LOCUM_ABS_FTE: 219 non-numeric values
TOTAL_GP_LOCUM_ABS_FTE
ND    219
Name: count, dtype: int64 

YEAR_MONTH
202207    25
202208    26
202209    31
202210    33
202211    35
Name: count, dtype

## Process

In [18]:
use_cols = id_cols + gp_fte_cols

dfs = []
for f in keep:
    y, m = file_period(f.name)
    df = pd.read_csv(f, usecols=use_cols, dtype={c: str for c in id_cols})
    df.insert(0, "YEAR_MONTH", f"{y}{m:02d}")
    dfs.append(df)

gpwf_slim = pd.concat(dfs, ignore_index=True)

gpwf_slim['YEAR_MONTH_DATE'] = pd.to_datetime(gpwf_slim['YEAR_MONTH'].astype(str), format='%Y%m')
gpwf_slim['YEAR'] = gpwf_slim['YEAR_MONTH_DATE'].dt.year
gpwf_slim['QUARTER'] = gpwf_slim['YEAR_MONTH_DATE'].dt.quarter
gpwf_slim = gpwf_slim.sort_values(['YEAR_MONTH_DATE', 'PRAC_CODE']).reset_index(drop=True)

print(gpwf_slim.shape)
print("months:", gpwf_slim["YEAR_MONTH"].nunique())
print("duplicate practice-months:", gpwf_slim.duplicated(["YEAR_MONTH", "PRAC_CODE"]).sum())
print(gpwf_slim[['YEAR_MONTH', 'YEAR_MONTH_DATE', 'YEAR', 'QUARTER']].drop_duplicates().head(8))


(283247, 32)
months: 45
duplicate practice-months: 0
      YEAR_MONTH YEAR_MONTH_DATE  YEAR  QUARTER
0         202207      2022-07-01  2022        3
6471      202208      2022-08-01  2022        3
12930     202209      2022-09-01  2022        3
19386     202210      2022-10-01  2022        4
25827     202211      2022-11-01  2022        4
32256     202212      2022-12-01  2022        4
38678     202301      2023-01-01  2023        1
45096     202302      2023-02-01  2023        1


In [19]:
df = pd.read_csv(f, usecols=use_cols, dtype={c: str for c in id_cols}, na_values=["ND"])

# 1. Confirm the FTE columns actually converted to numeric
fte_cols = [c for c in gpwf_slim.columns if c.endswith('_FTE')]
print(gpwf_slim[fte_cols].dtypes)

before = gpwf_slim[fte_cols].isna().sum()
for c in fte_cols:
    gpwf_slim[c] = pd.to_numeric(gpwf_slim[c], errors='coerce')
print((gpwf_slim[fte_cols].isna().sum() - before).sum(), "values turned into NaN")

# Does "Unaligned" (or similar placeholders) appear elsewhere?
for col in ['PCN_CODE', 'SUB_ICB_CODE', 'ICB_CODE', 'SUB_ICB_NAME', 'PCN_NAME', 'ICB_NAME']:
    if col in gpwf_slim.columns:
        placeholder_like = gpwf_slim[col].astype(str).str.contains(
            'unaligned|not assigned|not applicable|n/a|unknown', case=False, na=False
        )
        n = placeholder_like.sum()
        if n > 0:
            print(f"{col}: {n} rows with placeholder-like values")
            print(gpwf_slim.loc[placeholder_like, col].value_counts())

gpwf_slim['PCN_CODE'] = gpwf_slim['PCN_CODE'].replace('Unaligned', None)
gpwf_slim['PCN_NAME'] = gpwf_slim['PCN_NAME'].replace('Unaligned', None)

print(gpwf_slim[['PCN_CODE', 'PCN_NAME']].isna().sum())

TOTAL_GP_FTE                 object
TOTAL_GP_EXTG_FTE            object
TOTAL_GP_EXL_FTE             object
TOTAL_GP_EXTGL_FTE           object
TOTAL_GP_SEN_PTNR_FTE        object
TOTAL_GP_PTNR_PROV_FTE       object
TOTAL_GP_SAL_BY_PRAC_FTE     object
TOTAL_GP_SAL_BY_OTH_FTE      object
TOTAL_GP_TRN_GR_ST1_FTE     float64
TOTAL_GP_TRN_GR_ST2_FTE     float64
TOTAL_GP_TRN_GR_ST3_FTE     float64
TOTAL_GP_TRN_GR_ST4_FTE     float64
TOTAL_GP_TRN_GR_OTH_FTE     float64
TOTAL_GP_TRN_GR_F1_2_FTE    float64
TOTAL_GP_RET_FTE             object
TOTAL_GP_LOCUM_VAC_FTE       object
TOTAL_GP_LOCUM_ABS_FTE       object
TOTAL_GP_LOCUM_OTH_FTE       object
dtype: object
2624 values turned into NaN
PCN_CODE: 3458 rows with placeholder-like values
PCN_CODE
Unaligned    3458
Name: count, dtype: int64
PCN_NAME: 3458 rows with placeholder-like values
PCN_NAME
Unaligned    3458
Name: count, dtype: int64
PCN_CODE    3458
PCN_NAME    3458
dtype: int64


In [20]:
output_path = PROCESSED_DIR / "gpwf_processed.parquet"

gpwf_slim.to_parquet(
    output_path,
    engine="pyarrow",
    index=False,
    compression="snappy"
)

print(f"File size: {os.path.getsize(output_path) / 1e9:.2f} GB")

File size: 0.01 GB
